1. Загрузка JSON

In [108]:
import json
import pandas as pd

FILE_NAME = "raw_20260305_102335"
with open(f"../data/raw/{FILE_NAME}.json") as f:
    data = json.load(f)

2. Проверка структуры

In [109]:
type(data)

list

In [110]:
len(data)

2

In [111]:
type(data[0])

dict

In [112]:
type(data[1])

list

3. Построение DataFrame

In [113]:
df = pd.json_normalize(data[1])

4. Проверка таблицы

In [114]:
df.head()

,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,RUS,2025,NaN,,,0,NY.GDP.MKTP.CD,GDP (current US$),RU,Russian Federation
1,RUS,2024,2.173836e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),RU,Russian Federation
2,RUS,2023,2.071506e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),RU,Russian Federation
3,RUS,2022,2.291612e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),RU,Russian Federation
4,RUS,2021,1.829187e+12,,,0,NY.GDP.MKTP.CD,GDP (current US$),RU,Russian Federation


In [115]:
df.shape

(66, 10)

In [116]:
df.columns

Index(['countryiso3code', 'date', 'value', 'unit', 'obs_status', 'decimal',
       'indicator.id', 'indicator.value', 'country.id', 'country.value'],
      dtype='object')

In [117]:
df.dtypes


countryiso3code     object
date                object
value              float64
unit                object
obs_status          object
decimal              int64
indicator.id        object
indicator.value     object
country.id          object
country.value       object
dtype: object

5. Очистка данных

In [118]:
df = df.rename(columns={
    "indicator.id": "indicator_id",
    "indicator.value": "indicator_name",
    "country.id": "country_id",
    "country.value": "country_name"
})

df["date"] = pd.to_datetime(df["date"], format="%Y")

df["value"] = pd.to_numeric(df["value"], errors="coerce")

df = df[
    [
        "country_id",
        "country_name",
        "countryiso3code",
        "indicator_id",
        "indicator_name",
        "date",
        "value"
    ]
]

df = df.dropna(subset=["value"])
df = df.drop_duplicates()

6. Проверка после очистки

In [119]:
df.head()

,country_id,country_name,countryiso3code,indicator_id,indicator_name,date,value
1,RU,Russian Federation,RUS,NY.GDP.MKTP.CD,GDP (current US$),2024-01-01,2.173836e+12
2,RU,Russian Federation,RUS,NY.GDP.MKTP.CD,GDP (current US$),2023-01-01,2.071506e+12
3,RU,Russian Federation,RUS,NY.GDP.MKTP.CD,GDP (current US$),2022-01-01,2.291612e+12
4,RU,Russian Federation,RUS,NY.GDP.MKTP.CD,GDP (current US$),2021-01-01,1.829187e+12
5,RU,Russian Federation,RUS,NY.GDP.MKTP.CD,GDP (current US$),2020-01-01,1.493076e+12


In [120]:
df.dtypes

country_id                 object
country_name               object
countryiso3code            object
indicator_id               object
indicator_name             object
date               datetime64[ns]
value                     float64
dtype: object

In [121]:
df.isna().sum()

country_id         0
country_name       0
countryiso3code    0
indicator_id       0
indicator_name     0
date               0
value              0
dtype: int64

In [122]:
df.shape

(37, 7)

7. Сохранение normalized CSV

In [123]:
df.to_csv(
    f"../data/normalized/{FILE_NAME}.csv",
    index=False
)